-----

# **Capstone Project: Building an AI Assistant Specialized in RAG with OpenAI**

## **Introduction**

Welcome to our final lesson and the Capstone project! Here, we will consolidate all the knowledge acquired to build an end-to-end Retrieval-Augmented Generation (RAG) system.

**The Problem:** The education company Alura wants to create an intelligent chatbot to assist prospective students. This assistant must answer specific questions about the RAG course content, using the course PDFs as its knowledge base. The goal is to provide accurate and reliable answers based solely on the course material.

**The Solution:** We will build a complete RAG system, following the steps and using the tools discussed in the lessons.

This notebook will guide you through each step:

1.  **Environment Setup:** Installing libraries and configuring the OpenAI API key.
2.  **Phase 1: Data Ingestion Pipeline (ETL)**
      * **Extraction (Extract):** Load the course PDF documents.
      * **Transformation (Transform):** Apply adaptive chunking strategies and generate embeddings.
      * **Loading (Load):** Index the chunks and their embeddings into a vector database, Chroma.
3.  **Phase 2: Building an Advanced Retrieval System**
      * Implement **Hybrid Search** to combine lexical (BM25) and semantic search.
4.  **Phase 3: Creating a Robust Conversational Chain**
      * Implement the `ConversationalRetrievalChain`.
      * Integrate **memory management** to maintain dialogue context.
      * Apply **Query Transformation**, which occurs internally in the chain to refine questions based on history.
5.  **Phase 4: System Evaluation with RAGAS**
      * Evaluate the system's performance using RAGAS-specific metrics, such as *Faithfulness*, *Answer Relevancy*, *Context Precision*, and *Context Recall*.

Let's get started\!

-----

### **1. Environment Setup**

First, let's install the necessary libraries and configure the API key.

In [1]:
!pip install -q langchain langchain-core langchain-community langchain-openai

!pip install -q openai python-dotenv

!pip install -q chromadb pypdf rank_bm25

!pip install -q ragas datasets

In [2]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv(dotenv_path='../../.env')

# Verify that the OpenAI API key is loaded
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print("✅ OpenAI API key loaded successfully.")
else:
    print("❌ Failed to load OpenAI API key. Please check your .env file.")
    print("Make sure your .env file contains: OPENAI_API_KEY=your_key_here")

✅ OpenAI API key loaded successfully.


### **2. Phase 1: Data Ingestion Pipeline (ETL)**

Let's structure our knowledge base for the RAG system.

#### **2.1 Extraction (Extract)**

We will use `Document Loaders` from LangChain to load the PDFs.

*Simulation: Upload all course PDFs to a directory named `course_documents`.*

In [3]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("../../data", glob="*.pdf")
docs = loader.load()

/Users/julio.cesar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/julio.cesar/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
len(docs)

7

#### **2.2 Transformation (Transform)**

Now, we will split the documents into chunks and convert them into embeddings.

**Adaptive Chunking:** We will use the `RecursiveCharacterTextSplitter` to intelligently split the text.

In [5]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(docs)

print(f"Total number of chunks created: {len(chunks)}")

Total number of chunks created: 15


**High-Performance Embeddings:** The quality of the embeddings is crucial. We will use OpenAI's `text-embedding-3-small` model, which provides excellent performance and cost-effectiveness.

In [6]:
from langchain_openai import OpenAIEmbeddings

# Use OpenAI's embedding model for high-quality embeddings
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # Cost-effective and performant
    openai_api_key=openai_api_key
)

#### **2.3 Loading (Load)**

We index the chunks and their embeddings into a vector database. We will use **Chroma** for this.

Create a vector database in Chroma by indexing the chunks with the embeddings_model, returning the vectorstore object.
Then print a message confirming that the database was successfully created.

In [7]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model)

print("Database created successfully!")

Database created successfully!


### **3. Phase 2: Building an Advanced Retrieval System**

Let's create a retriever that uses hybrid search to find the best documents.

#### **3.1 Hybrid Search**

Hybrid search combines semantic search (vector-based) with lexical search (BM25). This leverages the semantic search's context understanding and the lexical search's exact term precision.

In [8]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

# 1. Lexical Retriever (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# 2. Vector Retriever (from our ChromaDB)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})


# 3. EnsembleRetriever to combine results
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

print("Hybrid Search Retriever Configured")

Hybrid Search Retriever Configured


### **4. Phase 3: Creating a Robust Conversational Chain**

Now, let's build the chain that orchestrates the interaction with the user, using Gemini as the brain of our chatbot.

#### **4.1 Chain with Memory and Query Transformation**

The `ConversationalRetrievalChain` is perfect for this use case. It manages the conversation history to maintain context and rewrites the user's question to make it autonomous, a form of **Query Transformation**.

In [9]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# Initialize the LLM (GPT-4o-mini is cost-effective and powerful)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key=openai_api_key,
    temperature=0.3,  # Lower temperature for more focused, factual responses
)

# Set up conversation memory to maintain context across exchanges
memory = ConversationBufferMemory(
    memory_key="chat_history",
    output_key="answer",
    return_messages=True
)

# Create the conversational RAG chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=ensemble_retriever,
    memory=memory,
    verbose=False,
    return_source_documents=True
)

print("✅ Conversational RAG chain configured successfully!")


✅ Conversational RAG chain configured successfully!


/var/folders/v3/j1dtbpl50r9_jp0s85h7c3lw0000gp/T/ipykernel_93784/4081225787.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [10]:
response1 = qa_chain.invoke({"question": "What is adaptive chunking?"})
print(response1['answer'])

I don't know.


In [11]:
response2 = qa_chain.invoke({"question": "And what are the main strategies?"})
print(response2['answer'])

I don't know.


### **5. Phase 4: System Evaluation with RAGAS**

Evaluating our system is essential to ensure its quality. We will use **RAGAS**, a library that provides metrics specific to RAG systems.

We will evaluate based on the **Generation** metrics (*Faithfulness*, *Answer Relevancy*) and **Retrieval** metrics (*Context Precision*, *Context Recall*).

In [12]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

# 1. Create a dataset for evaluation
questions = [
    "What is RAG and what problem does it solve?",
    "What are the essential components of RAG?",
    "What is the difference between lexical and semantic search?",
    "What does the faithfulness metric of RAGAS measure?"
]
golden_answers = [
    "RAG (Retrieval-Augmented Generation) is an architecture that combines a search engine to retrieve information with a language model (LLM) to generate answers. It solves problems like hallucinations and outdated knowledge of LLMs.",
    "The essential components are: Embeddings, Vector Database, Chunking, and a Language Model (LLM).",
    "Lexical search (like BM25) finds exact term matches, while semantic search captures meaning and context, even with different words.",
    "The Faithfulness metric measures if the generated answer is supported and factually consistent with the retrieved documents, avoiding hallucinations."
]

# 2. Generate answers and contexts with our chain
print("🔄 Generating answers for evaluation...")
generated_answers = []
retrieved_contexts = []
for i, question in enumerate(questions, 1):
    print(f"   Processing question {i}/{len(questions)}...")
    result = qa_chain.invoke({"question": question})
    generated_answers.append(result['answer'])
    retrieved_contexts.append([doc.page_content for doc in result['source_documents']])

# 3. Create the dataset in the format expected by RAGAS
dataset_dict = {
    'question': questions,
    'answer': generated_answers,
    'contexts': retrieved_contexts,
    'ground_truth': golden_answers
}
dataset = Dataset.from_dict(dataset_dict)

# 4. Run the evaluation using OpenAI models
print("\n📊 Running RAGAS evaluation...")
evaluation_result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=llm,  # Use the same LLM we configured earlier
    embeddings=embeddings_model  # Use the same embeddings model
)

# 5. Analyze the results
df_results = evaluation_result.to_pandas()
print("\n✅ Evaluation Results with RAGAS:")
print("="*80)
display(df_results)
print("\n📈 Average Scores:")
print(f"   Faithfulness:       {df_results['faithfulness'].mean():.3f}")
print(f"   Answer Relevancy:   {df_results['answer_relevancy'].mean():.3f}")
print(f"   Context Precision:  {df_results['context_precision'].mean():.3f}")
print(f"   Context Recall:     {df_results['context_recall'].mean():.3f}")

🔄 Generating answers for evaluation...
   Processing question 1/4...
   Processing question 2/4...
   Processing question 3/4...
   Processing question 4/4...

📊 Running RAGAS evaluation...


Evaluating:   6%|▋         | 1/16 [00:03<00:56,  3.77s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 16/16 [00:55<00:00,  3.47s/it]



✅ Evaluation Results with RAGAS:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is RAG and what problem does it solve?,[Building Intelligent Chatbots with \nRetr...,Retrieval-Augmented Generation (RAG) is an arc...,RAG (Retrieval-Augmented Generation) is an arc...,0.923077,0.569167,0.916667,1.0
1,What are the essential components of RAG?,[Building Intelligent Chatbots with \nRetr...,The essential components of a Retrieval-Augmen...,"The essential components are: Embeddings, Vect...",1.000000,0.661852,0.250000,1.0
2,What is the difference between lexical and sem...,"[essential, where the strategy changes ba...",I don't know.,Lexical search (like BM25) finds exact term ma...,0.000000,0.000000,1.000000,1.0
3,What does the faithfulness metric of RAGAS mea...,[Evaluating a RAG system is complex beca...,The faithfulness metric of RAGAS measures how ...,The Faithfulness metric measures if the genera...,1.000000,0.999999,0.750000,1.0



📈 Average Scores:
   Faithfulness:       0.731
   Answer Relevancy:   0.558
   Context Precision:  0.729
   Context Recall:     1.000
